# Phase 6: Industrial AI Systems - Task 1
## The "Corporate Office" (Multi-Agent Setup)

### **Goal**
In this notebook, we implement **Module 1: Multi-Agent Architectures**. We will build a 3-agent team that works autonomously to handle a business request:
1. **Agent A (Researcher):** Responsible for data discovery.
2. **Agent B (Writer):** Responsible for summarizing that data into a draft.
3. **Agent C (Manager):** Responsible for quality control and triggering **Self-Correction Loops**

In [1]:
# Install necessary orchestration libraries
# !pip install langgraph langchain_openai

import operator
from typing import Annotated, TypedDict, List
from langgraph.graph import StateGraph, END

c:\Users\zarya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


### **Defining Shared Memory (State)**
A key part of **Agentic Orchestration** is the "Handoff Protocol". We use a `TypedDict` to act as the **Shared Memory** (State), ensuring every agent knows what the previous agent discovered.

In [2]:
class AgentState(TypedDict):
    topic: str
    research_notes: str
    report: str
    review_feedback: str
    revision_count: int  # Track iterations for the loop

### **Designing Specialized Personas**
We now define the specialized personas for our team. Each function represents an agent's "Action Logic":
- **Researcher:** Simulates data gathering.
- **Writer:** Generates the report.
- **Manager:** Audits the report against quality standards (length and detail).

In [3]:
def researcher_agent(state: AgentState):
    print("\n--- AGENT A (RESEARCHER): GATHERING DATA ---")
    # Simulate a tool discovery phase
    notes = f"Research findings for {state['topic']}: Data Point 1 (Specs), Data Point 2 (Market Trend)."
    return {"research_notes": notes}

def writer_agent(state: AgentState):
    print("--- AGENT B (WRITER): DRAFTING REPORT ---")
    # In a real scenario, this would use an LLM call
    draft = f"Topic: {state['topic']}\nFindings: {state['research_notes']}"
    return {"report": draft}

def manager_agent(state: AgentState):
    print("--- AGENT C (MANAGER): PERFORMING QUALITY AUDIT ---")
    # Logic for Self-Correction: If the report is too short, reject it
    if len(state['report']) < 80:
        feedback = "REJECTED: The report is too brief. Please expand on the technical specs."
        return {"review_feedback": feedback}
    return {"review_feedback": "APPROVED"}

### **Building the Orchestrator**
We use **LangGraph** to define the flow. The "Manager" agent acts as the decision-maker, routing the workflow either to the **END** or back to the **Writer** for a revision.

In [4]:
def routing_logic(state: AgentState):
    """Determines if we finish or loop back for correction"""
    if state["review_feedback"] == "APPROVED":
        return "end"
    return "rewrite"

# Initialize Graph
workflow = StateGraph(AgentState)

# Add Nodes (The agents)
workflow.add_node("researcher", researcher_agent)
workflow.add_node("writer", writer_agent)
workflow.add_node("manager", manager_agent)

# Define Connections (Edges)
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "manager")

# Add the Self-Correction Loop (Conditional Edge)
workflow.add_conditional_edges(
    "manager",
    routing_logic,
    {"rewrite": "writer", "end": END}
)

app = workflow.compile()

### **Running the System**
We will now trigger the system with a topic. Observe the terminal log to see the agents "talking" and the Manager enforcing the quality loop.

In [5]:
# Set initial state
initial_input = {
    "topic": "Industrial AI Agents in 2026", 
    "revision_count": 0,
    "report": ""
}

# Run the autonomous team
for output in app.stream(initial_input):
    for key, value in output.items():
        if "review_feedback" in value:
            print(f"Manager Status: {value['review_feedback']}")
    print("-" * 30)


--- AGENT A (RESEARCHER): GATHERING DATA ---
------------------------------
--- AGENT B (WRITER): DRAFTING REPORT ---
------------------------------
--- AGENT C (MANAGER): PERFORMING QUALITY AUDIT ---
Manager Status: APPROVED
------------------------------
